# 07 - Model-Version Tag Propagation Demo

Companion notebook to `../08-production-resilience-and-operational-engineering.md`'s concurrency
caveat: *"a model-artifact cache that outlives its own 'new model deployed' signal."* The chapter's
scenario -- a Lambda routing layer that reads the current model-version pointer once per execution
environment and caches it for that environment's lifetime, so a retraining promotion doesn't reach
every warm environment simultaneously -- is simulated here end to end: an SSM-style pointer, multiple
Lambda execution environments that cache it at different times, a mixed-version batch of classification
results, and the fix chapter 08 names directly: *"stamp every result with the model version actually
used for that specific invocation."*

Fully offline -- only the standard library and `numpy`. No AWS, no Sagemaker, no internet.

In [1]:
import numpy as np
import time

np.random.seed(42)
print("Ready.")

Ready.


## 1. A tiny classifier stand-in, version-dependent on purpose

Real code invokes a Sagemaker multi-model endpoint's `target_model` path (chapter 05). Here,
`classify_claim` is a small deterministic stand-in whose output can differ by `model_version` -- this
matters for the demo: if every model version produced identical output, a mixed-version window would be
invisible even *with* tagging, because there would be nothing for the tag to explain. Version `v2` is a
retrained model (chapter 04) that reclassifies a specific claim pattern differently than `v1` did --
mirroring chapter 08 bug #2's "a tag list change shifted classifications" flavor of real-world drift,
without literally reproducing that bug.

In [2]:
LABELS_BY_VERSION = {
    "v1": {
        "reduces symptom severity": "efficacy",
        "cardiovascular risk": "safety",
        "starting dose": "dosing",
    },
    "v2": {
        # v2 was retrained after a claim-type tag split (chapter 08 bug #2's scenario) -- comparative
        # efficacy claims naming a specific percentage are now tagged more precisely.
        "reduces symptom severity": "efficacy-comparative",
        "cardiovascular risk": "safety",
        "starting dose": "dosing",
    },
}


def classify_claim(text: str, model_version: str) -> str:
    """Deterministic stand-in for a Sagemaker invoke_endpoint(target_model=...) call. Returns the
    first matching label for the resolved model_version, or "unclassified" if nothing matches."""
    rules = LABELS_BY_VERSION[model_version]
    lower = text.lower()
    for pattern, label in rules.items():
        if pattern in lower:
            return label
    return "unclassified"


print(classify_claim("Drug X reduces symptom severity by 38% vs placebo.", "v1"))
print(classify_claim("Drug X reduces symptom severity by 38% vs placebo.", "v2"))

efficacy
efficacy-comparative


## 2. The SSM-style pointer, and the buggy cached-at-cold-start Lambda environment

`ssm_current_model_version` stands in for the SSM Parameter Store value chapter 05/08 describe --
whatever the deployment step most recently pointed the endpoint's `target_model` at. `BuggyLambdaEnv`
reproduces the caveat exactly: it reads the pointer **once, at construction** ("cold start"), caches it
in an instance variable, and reuses that cached value for every `invoke()` call for as long as the
environment stays "warm" -- it never re-reads the pointer on its own.

In [3]:
ssm_current_model_version = {"value": "v1"}  # the deployment step's source of truth


class BuggyLambdaEnv:
    """Reproduces chapter 08's caveat: reads the model-version pointer once at cold start, caches it
    for the environment's lifetime, and never re-checks it -- even after a new version is promoted."""

    def __init__(self, env_id: str):
        self.env_id = env_id
        self.cached_model_version = ssm_current_model_version["value"]  # read ONCE, at cold start

    def invoke(self, claim_text: str) -> dict:
        label = classify_claim(claim_text, self.cached_model_version)
        return {
            "env_id": self.env_id,
            "claim_text": claim_text,
            "label": label,
            "model_version": self.cached_model_version,  # stamped -- see section 4 for why this matters
        }


# Two execution environments come up (cold start) while the endpoint is still on v1.
env_a = BuggyLambdaEnv("lambda-env-A")
env_b = BuggyLambdaEnv("lambda-env-B")
print(env_a.env_id, "cached:", env_a.cached_model_version)
print(env_b.env_id, "cached:", env_b.cached_model_version)

lambda-env-A cached: v1
lambda-env-B cached: v1


## 3. A retraining pipeline promotes v2 -- but warm environments don't find out

The deployment step (chapter 04) repoints the endpoint at the newly promoted `v2`. `env_a` and `env_b`
are already warm and have no mechanism to notice -- they keep using their cached `v1` pointer. A brand
new environment (`env_c`), cold-started *after* the promotion, correctly picks up `v2`. This is exactly
chapter 08's "two content items submitted seconds apart... can be classified by two different model
versions" scenario.

In [4]:
# The retraining pipeline promotes v2 (chapter 04's "repoint the endpoint at a new registered version").
ssm_current_model_version["value"] = "v2"
print("SSM pointer is now:", ssm_current_model_version["value"])

# env_a and env_b are still warm from before the promotion -- their cached pointer is stale.
print("env_a cached model_version (unchanged):", env_a.cached_model_version)
print("env_b cached model_version (unchanged):", env_b.cached_model_version)

# A brand new execution environment cold-starts AFTER the promotion and correctly picks up v2.
env_c = BuggyLambdaEnv("lambda-env-C")
print("env_c cached model_version (fresh cold start):", env_c.cached_model_version)

SSM pointer is now: v2
env_a cached model_version (unchanged): v1
env_b cached model_version (unchanged): v1
env_c cached model_version (fresh cold start): v2


## 4. A mixed-version batch, tagged -- the fix chapter 08 names

Requests for the same claim text, load-balanced across `env_a`/`env_b` (stale, still `v1`) and `env_c`
(fresh, `v2`), arrive seconds apart during the promotion window. Because `invoke()` stamps every result
with `self.cached_model_version` -- the version *actually used for that specific invocation*, not the
value the deployment step *intended* to be live -- the mixed-version window becomes something you can
directly query, exactly as chapter 08 describes: *"you can query 'how many of the last hour's results
came from the outgoing version' instead of having no way to even ask the question."*

In [5]:
claim_text = "Drug X reduces symptom severity by 38% vs placebo."

# Requests hit whichever environment the load balancer happened to route them to during the window.
batch = [
    env_a.invoke(claim_text),
    env_c.invoke(claim_text),
    env_b.invoke(claim_text),
    env_c.invoke(claim_text),
    env_a.invoke(claim_text),
]

for r in batch:
    print(f"{r['env_id']:14s}  model_version={r['model_version']}  label={r['label']!r}")

lambda-env-A    model_version=v1  label='efficacy'
lambda-env-C    model_version=v2  label='efficacy-comparative'
lambda-env-B    model_version=v1  label='efficacy'
lambda-env-C    model_version=v2  label='efficacy-comparative'
lambda-env-A    model_version=v1  label='efficacy'


In [6]:
def detect_mixed_version_window(results: list[dict], current_version: str) -> dict:
    """The detection query chapter 08 says becomes directly answerable once every result carries its
    producing model_version: how many results in this batch came from a version other than the
    currently-intended one, and which specific results were they?"""
    by_version = {}
    for r in results:
        by_version.setdefault(r["model_version"], []).append(r)

    stale_versions = {v: rows for v, rows in by_version.items() if v != current_version}
    return {
        "counts_by_version": {v: len(rows) for v, rows in by_version.items()},
        "outgoing_version_count": sum(len(rows) for rows in stale_versions.values()),
        "outgoing_version_results": stale_versions,
    }


report = detect_mixed_version_window(batch, current_version=ssm_current_model_version["value"])
print("Counts by model_version:", report["counts_by_version"])
print("Results still served by the outgoing version:", report["outgoing_version_count"], "of", len(batch))

assert report["outgoing_version_count"] == 3, "expected env_a's two and env_b's one stale-v1 invocations"
print("\nConfirmed: the mixed-version window is directly queryable and quantifiable, not invisible.")

Counts by model_version: {'v1': 3, 'v2': 2}
Results still served by the outgoing version: 3 of 5

Confirmed: the mixed-version window is directly queryable and quantifiable, not invisible.


## 5. The counterfactual: the same batch, without version tagging

This is the point chapter 08 makes explicit: tagging doesn't shrink the staleness window at all -- it
makes the window *observable*. To show the difference concretely, the cell below reruns an equivalent
batch through a classifier that returns only the label, with no `model_version` field, and shows that
the question "how many results came from the outgoing version" simply cannot be answered from the
untagged results, even though the underlying mixed-version behavior is identical.

In [7]:
class BuggyLambdaEnvUntagged(BuggyLambdaEnv):
    """Same caching bug as BuggyLambdaEnv, but the result does NOT carry model_version -- the
    "invisible mixed-version window" state chapter 08 contrasts with the fix."""

    def invoke(self, claim_text: str) -> dict:
        label = classify_claim(claim_text, self.cached_model_version)
        return {"env_id": self.env_id, "claim_text": claim_text, "label": label}  # no model_version


untagged_env_a = BuggyLambdaEnvUntagged("lambda-env-A")
untagged_env_c = BuggyLambdaEnvUntagged("lambda-env-C")
untagged_env_a.cached_model_version = "v1"  # still-warm, stale
untagged_env_c.cached_model_version = "v2"  # freshly cold-started, current

untagged_batch = [
    untagged_env_a.invoke(claim_text),
    untagged_env_c.invoke(claim_text),
    untagged_env_a.invoke(claim_text),
]

print("Untagged batch:")
for r in untagged_batch:
    print(" ", r)

can_answer_the_question = all("model_version" in r for r in untagged_batch)
print("\nCan we tell which results came from the outgoing model version?", can_answer_the_question)
assert not can_answer_the_question, "untagged results must not expose which model version produced them"
print("Confirmed: without the model_version tag, the same mixed-version window happened, but there is")
print("no way to ask -- let alone answer -- 'how many of these were served by the outgoing version.'")

Untagged batch:
  {'env_id': 'lambda-env-A', 'claim_text': 'Drug X reduces symptom severity by 38% vs placebo.', 'label': 'efficacy'}
  {'env_id': 'lambda-env-C', 'claim_text': 'Drug X reduces symptom severity by 38% vs placebo.', 'label': 'efficacy-comparative'}
  {'env_id': 'lambda-env-A', 'claim_text': 'Drug X reduces symptom severity by 38% vs placebo.', 'label': 'efficacy'}

Can we tell which results came from the outgoing model version? False
Confirmed: without the model_version tag, the same mixed-version window happened, but there is
no way to ask -- let alone answer -- 'how many of these were served by the outgoing version.'


## 6. The complementary hardening fix: bound the staleness window with a TTL

Chapter 08 names tagging as the fix that matters most (observability over elimination), but pairs it
with a cheap, optional second fix: give the cached pointer a short TTL so even a long-lived warm
environment re-checks it periodically, rather than only on cold start. `TTLAwareLambdaEnv` below adds
that on top of the tagging fix -- both hardenings composed together.

In [8]:
class TTLAwareLambdaEnv(BuggyLambdaEnv):
    """Adds a short TTL to the cached pointer: even a long-warm environment re-reads SSM after
    `ttl_seconds`, bounding the staleness window instead of relying on the environment's own,
    traffic-dependent recycling schedule."""

    def __init__(self, env_id: str, ttl_seconds: float = 2.0):
        super().__init__(env_id)
        self.ttl_seconds = ttl_seconds
        self._cached_at = time.monotonic()

    def _maybe_refresh(self) -> None:
        if time.monotonic() - self._cached_at >= self.ttl_seconds:
            self.cached_model_version = ssm_current_model_version["value"]
            self._cached_at = time.monotonic()

    def invoke(self, claim_text: str) -> dict:
        self._maybe_refresh()
        return super().invoke(claim_text)


ttl_env = TTLAwareLambdaEnv("lambda-env-D", ttl_seconds=0.05)
ttl_env.cached_model_version = "v1"  # simulate: cold-started before the promotion, cached v1

print("Immediately after cold start, before TTL elapses:", ttl_env.invoke(claim_text)["model_version"])

time.sleep(0.1)  # exceed the TTL

print("After the TTL window elapses, next invoke() self-heals:", ttl_env.invoke(claim_text)["model_version"])
assert ttl_env.invoke(claim_text)["model_version"] == ssm_current_model_version["value"]
print("\nConfirmed: a TTL bounds the staleness window to a known, small duration instead of leaving it")
print("dependent on however long Lambda happens to keep that execution environment warm.")

Immediately after cold start, before TTL elapses: v1
After the TTL window elapses, next invoke() self-heals: v2

Confirmed: a TTL bounds the staleness window to a known, small duration instead of leaving it
dependent on however long Lambda happens to keep that execution environment warm.


## Takeaways

- **The bug is a caching lifecycle mismatch, not a logic error.** `BuggyLambdaEnv` correctly reads the
  SSM pointer and correctly classifies claims -- the problem is purely *when* it reads the pointer
  (once, at cold start) versus when the pointer actually changes (whenever a retraining pipeline
  promotes a new version), and those two events are completely decoupled (Sections 2-3).
- **Stamping every result with `model_version` doesn't shrink the mixed-version window -- it makes the
  window queryable.** Section 4's `detect_mixed_version_window` turns "did this happen, and how much"
  from an unanswerable question into a two-line aggregation. Section 5 shows the same underlying bug
  with the tag removed: identical behavior, but the question can no longer even be asked.
- **A TTL on the cached pointer is a complementary, not competing, fix.** It bounds *how long* the
  staleness window can be; the tag makes that window *observable* regardless of how long it turns out to
  be. Chapter 08's framing is deliberate: the fix that matters most for a compliance pipeline is
  observability, because eliminating the window entirely would mean fighting Lambda's own
  execution-environment lifecycle, which is expensive and never fully guaranteed anyway.